<a href="https://colab.research.google.com/github/SuryamaheshPenke/Gen_AI-Notes/blob/main/Class_7_Chain_of_Thought_and_Reasoning_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Class 7 - Chain-of-Thought and Reasoning Techniques**
### Module 1: Prompt Engineering & LLM Mastery
---

## **Agenda**

- Why CoT was invented - the multi-step reasoning failure and the 2022 paper
- Standard CoT - step-by-step reasoning and scratchpad patterns
- Self-consistency - multiple reasoning paths and majority voting
- Tree of Thought - branching exploration for planning problems
- Reflection and self-critique - model reviews its own output
- Choosing the right technique for a given problem
- Connection back to reasoning models (Class 3: o1/o3/Claude Extended Thinking)

## **PART 1 - The Problem That CoT Solves**

In 2021, large language models were getting impressive scores on benchmarks - but they were failing in a specific, consistent way. Ask GPT-3 a multi-step math word problem or a logical puzzle, and it would give a confident, wrong answer with no work shown. The model seemed to know a lot of facts but could not reason through a chain of steps.

The key observation: LLMs are next-token predictors. When you ask "What is 17 × 24 + 89?", the model tries to predict the final answer token directly from the question tokens. There is no intermediate computation happening. It is pattern-matching the question to answers it has seen before.

This worked fine for one-step lookups. It broke down for anything requiring multiple dependent steps - math, logic chains, code reasoning, multi-hop question answering.

In January 2022, Wei et al. at Google published ["Chain-of-Thought Prompting Elicits Reasoning in Large Language Models"](https://arxiv.org/abs/2201.11903). The finding was simple and striking: if you show the model examples where the answer includes intermediate reasoning steps, the model starts producing intermediate steps too - and gets the right answer far more often.

No fine-tuning. No architecture change. Just a different prompt format.

This class covers CoT and everything that grew from it. By the end, you will know when to use each technique and why it works.

## **PART 2 - Standard Chain-of-Thought**

### What it is and why it works

**Simple**

Instead of asking the model "What is the answer?", you ask it to show its work first. You either give it examples with worked solutions (few-shot CoT) or you just add "Think step by step" to the prompt (zero-shot CoT, from Kojima et al. 2022).

The analogy: imagine asking someone to do a calculation in their head vs. on paper. On paper, each step anchors the next. The model's "paper" is the tokens it has already generated - once it writes step 1, step 2 is grounded in that output rather than floating in space.

**Technical**

In a transformer's forward pass, each generated token attends to all previous tokens in the context. When the model writes intermediate reasoning steps, those steps become part of the context that subsequent tokens attend to. The reasoning chain acts as a scratchpad - intermediate conclusions are now in the attention window and can directly condition the final answer token.

This is why CoT is an emergent behavior that requires sufficient model scale (~100B parameters in the original paper). Smaller models generate plausible-looking steps but the steps are not causally connected to the answer. At scale, the model has learned enough world knowledge and reasoning patterns that intermediate steps genuinely constrain the final output.

---

### When to use standard CoT

**Simple**

Use it whenever the problem has multiple steps where each step depends on the previous one. Math, logical deduction, code debugging, breaking down a complex question. If you could solve it in your head in one glance, CoT adds no value. If you would need a notepad, add CoT.

**Technical**

CoT improves performance on tasks that require symbolic reasoning, arithmetic, commonsense inference with multiple hops, and code generation where intermediate state matters. It does not help significantly on single-step tasks (direct retrieval, classification) and can introduce verbosity overhead. The scratchpad pattern - "Let me work through this:" followed by numbered steps - is the most reliable format because it signals to the model that a reasoning trace is expected before the final answer.

In [ ]:
# NOTE: We cover the OpenAI SDK, API keys, and Python environments properly in Class 13.
# This cell is provided as a reference only - do not run it during Class 7.
# The demos in this notebook are shown as static markdown examples instead.

# Demo setup - install dependencies
# !pip install openai anthropic --quiet

# import os
# from openai import OpenAI

# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# client = OpenAI()

# def ask(prompt, model='gpt-4o-mini', temperature=0):
#     response = client.chat.completions.create(
#         model=model,
#         messages=[{'role': 'user', 'content': prompt}],
#         temperature=temperature
#     )
#     return response.choices[0].message.content


## **Code Demos**

### **Demo 1 - Direct Answer vs Chain-of-Thought**

**Note: We cover the OpenAI SDK properly in Class 13. The prompts and outputs below are static examples so you can follow the concept now.**

---

**The same problem, two prompt styles:**

```
Problem:
A shop sells notebooks for 45 rupees each and pens for 12 rupees each.
Rohan buys 3 notebooks and some pens. He pays 201 rupees in total.
How many pens did he buy?
```

**Direct prompt:** `Answer:`

```
Sample model output:
5 pens.
```

**CoT prompt:** `Solve this step by step, then give the final answer.`

```
Sample model output:
Step 1: Cost of 3 notebooks = 3 x 45 = 135 rupees.
Step 2: Remaining amount for pens = 201 - 135 = 66 rupees.
Step 3: Number of pens = 66 / 12 = 5.5.

Wait - 5.5 is not a whole number. Re-checking: 3 x 45 = 135. 201 - 135 = 66. 66 / 12 = 5.5.
The CoT trace surfaces this ambiguity explicitly, whereas the direct answer just states a number
with no way to verify whether the model's arithmetic was correct.
```


Look at the direct answer - it may be correct or wrong, but notice there is no work shown. In the CoT output, the model calculates notebook cost (3 x 45 = 135), subtracts from total (201 - 135 = 66), then divides by pen price (66 / 12 = 5.5). Each step anchors the next. If the direct answer is wrong, the CoT version will typically be right and will show exactly where the reasoning diverges.

## **PART 3 - Self-Consistency**

### The problem with a single reasoning path

Standard CoT is better than direct answering, but it has a fragility: one unlucky reasoning path can produce a wrong answer confidently. The model commits to an approach in the first few tokens and follows it through, even if a different approach would have succeeded.

**Simple**

Instead of asking once, ask the same question multiple times with some temperature (randomness). Each run takes a slightly different reasoning path. Then look at all the final answers and pick the one that appears most often.

The analogy: five different engineers solve the same problem independently. Three get 42, one gets 41, one gets 43. You go with 42. You did not average - you voted.

**Technical**

Self-consistency (Wang et al. 2022, "Self-Consistency Improves Chain of Thought Reasoning in Language Models") samples k completions at temperature > 0, allowing the model to explore diverse reasoning trajectories. The final answer is determined by marginalizing over the reasoning paths - taking the plurality answer across all completions.

Formally: given question q, sample paths {r1, r2, ..., rk} and their answers {a1, a2, ..., ak}. Return argmax_a Count(ai = a). This works because correct reasoning paths tend to converge on the same answer even when they differ in approach, while errors are more likely to be diverse (different wrong answers). Self-consistency consistently outperforms greedy CoT by 2-10 percentage points on arithmetic and commonsense benchmarks, at the cost of k times the inference calls.

### **Demo 2 - Self-Consistency with Majority Vote**

**Note: We cover the OpenAI SDK and Python properly in Class 13. The reasoning paths below are static examples illustrating how self-consistency works.**

---

**Prompt:**

```
A train travels from city A to city B at 60 km/h.
The return journey is at 90 km/h.
What is the average speed for the entire round trip?
Think step by step.
```

**5 sampled reasoning paths:**

```
Path 1:
Let distance A->B = d km.
Time A->B = d/60. Time B->A = d/90.
Total distance = 2d. Total time = d/60 + d/90 = 3d/180 + 2d/180 = 5d/180.
Average speed = 2d / (5d/180) = 72 km/h.
Final answer: 72

Path 2:
Average speed = (60 + 90) / 2 = 75 km/h.
Final answer: 75

Path 3:
Using harmonic mean for equal distances: 2 x 60 x 90 / (60 + 90) = 10800/150 = 72 km/h.
Final answer: 72

Path 4:
Take d = 180 km. Time out = 3h. Time back = 2h.
Total distance = 360 km. Total time = 5h. Average speed = 72 km/h.
Final answer: 72

Path 5:
Average of 60 and 90 is 75.
Final answer: 75
```

**Vote distribution:** `{72: 3, 75: 2}`

**Self-consistency answer: 72** (appeared 3/5 times)

*(Correct answer: 72 km/h - harmonic mean, not arithmetic mean. Paths 2 and 5 fall into the common trap of averaging the two speeds directly.)*


This is a classic trap problem - students often average 60 and 90 to get 75, which is wrong. The correct answer is 72 km/h (harmonic mean of speeds). Watch whether the majority vote converges on 72 or 75. If some paths show 75 and others show 72, the correct paths outnumber the wrong ones - which is exactly what self-consistency relies on. Real improvement shows up most on problems where one-step shortcuts produce plausible but wrong results.

## **PART 4 - Tree of Thought**

### Beyond Linear Reasoning Chains

Both CoT and self-consistency follow linear paths - one step after another, or multiple paths voted on at the end. For some problems, the right approach is not linear. You need to explore branches, backtrack when a branch fails, and compare multiple partial solutions before committing. Tree of Thought (Yao et al. 2023) handles this class of problems.

**What it is:** ToT decomposes problem-solving into three components: a thought generator that produces k candidate next-steps from the current state, a state evaluator that scores each candidate (by prompting the model to judge "is this promising?"), and a search algorithm (BFS or DFS) that expands the most promising states. Unlike self-consistency which samples entire solutions and votes at the end, ToT prunes at each step.

**When to use it:** ToT is for problems where early decisions constrain later options and backtracking has real value - creative writing with strict constraints, game-like planning problems, multi-step code architecture decisions. For the 24-game (make 24 from 4 numbers), ToT achieves 74% success vs CoT's 4%. In practice, most real-world tasks do not require ToT - CoT or self-consistency handles them. ToT is a tool for the specific case where the solution space has a tree structure and intermediate state quality matters. The main downside: multiple LLM calls per decision point makes it significantly more expensive than standard CoT.

## **PART 5 - Reflection and Self-Critique**

### Making the model its own reviewer

All three techniques above happen before or during the reasoning. Reflection is different - it happens after. The model generates an answer, then reviews that answer against the original question, identifies weaknesses, and revises.

**Simple**

You write a first draft of an email. Then you re-read it and notice you forgot to mention the meeting time. You revise. Reflection is that re-read step, but done by the model on its own output.

This catches a specific class of errors: outputs that are internally coherent but miss something in the original requirement. CoT reasoning chains can follow correctly from a wrong assumption. Reflection gives the model a second pass to check assumptions against the original question.

**Technical**

Reflection (also called "self-refinement", Madaan et al. 2023) is a prompting pattern where you pass the model's initial output back as input with a critique prompt: "Review your answer above. Does it fully address the question? Are there errors, gaps, or unstated assumptions? Revise if needed."

This works because the model has different "behavior" in generation mode vs. evaluation mode. A model generating code may produce a subtle bug it would catch immediately when asked to review the same code. The mechanism is again attention: in the review pass, the original question tokens are equally available alongside the answer tokens, making it easier for the model to detect mismatches between what was asked and what was produced.

Reflection is particularly effective for: code correctness, factual claims, completeness of structured outputs, and any task with explicit success criteria you can state in the critique prompt.

### **Demo 4 - Reflection and Self-Critique**

**Note: We cover the OpenAI SDK properly in Class 13. The three passes below are static examples showing the generate-critique-revise loop.**

---

**Task:**

```
Write a Python function that takes a list of student scores
and returns a summary dict with: mean, median, highest, lowest,
and the count of students who scored above 75.
```

---

**Pass 1 - Initial generation:**

```python
def summarize_scores(scores):
    scores_sorted = sorted(scores)
    n = len(scores)
    mean = sum(scores) / n
    mid = n // 2
    median = scores_sorted[mid] if n % 2 != 0 else (scores_sorted[mid-1] + scores_sorted[mid]) / 2
    return {
        'mean': mean,
        'median': median,
        'highest': scores_sorted[-1],
        'lowest': scores_sorted[0],
        'above_75': sum(1 for s in scores if s > 75)
    }
```

---

**Pass 2 - Self-critique:**

```
Issues found:
1. Edge case: if scores is empty, len(scores) = 0 and sum(scores)/0 raises ZeroDivisionError.
2. All required keys are present: mean, median, highest, lowest, above_75. No missing keys.
Recommendation: add an empty-list guard at the top.
```

---

**Pass 3 - Revised output:**

```python
def summarize_scores(scores):
    if not scores:
        return {}
    scores_sorted = sorted(scores)
    n = len(scores)
    mean = sum(scores) / n
    mid = n // 2
    median = scores_sorted[mid] if n % 2 != 0 else (scores_sorted[mid-1] + scores_sorted[mid]) / 2
    return {
        'mean': round(mean, 2),
        'median': median,
        'highest': scores_sorted[-1],
        'lowest': scores_sorted[0],
        'above_75': sum(1 for s in scores if s > 75)
    }
```


Compare Pass 1 and Pass 3 carefully. The critique step often surfaces missing edge case handling (empty list raises ZeroDivisionError on mean calculation) and missing dictionary keys. The revised version typically adds `if not scores: return {}` guards and confirms all required keys are present. This is a reliable pattern for any output that has explicit completeness criteria - you can encode your requirements in the critique prompt and the model will check itself against them.

## **PART 6 - Connecting to Reasoning Models**

In Class 3, we covered reasoning models: o1, o3, and Claude Extended Thinking. Now that you understand CoT, the connection is direct.

**Simple**

Reasoning models are models that have been trained to do CoT internally, automatically, before every answer. You do not write "think step by step" - the model always thinks step by step. The internal reasoning tokens are called a "thinking trace" or "reasoning trace", and you sometimes get to see them.

**Technical**

Reasoning models (OpenAI o1/o3, Claude 3.7 Sonnet with extended thinking) are trained using reinforcement learning on reasoning traces rather than next-token prediction alone. The model learns to generate a chain of thought as a latent computation before producing the final answer. This trace uses a separate "thinking" token budget that does not count against your output tokens in the same way.

The key architectural difference from prompt-level CoT: in standard CoT prompting, the reasoning chain is in the main context window and can be seen/interrupted. In reasoning models, the chain is generated in a separate reasoning phase, and only the final answer is returned by default (you can request the trace). This is why reasoning models dramatically outperform standard CoT on competition math and complex multi-step tasks - the training signal reinforces reasoning quality, not just next-token plausibility.

From a practical standpoint: use reasoning models when the problem is hard enough that you would otherwise need to run self-consistency or ToT manually. They are slower and more expensive, but they automate the technique selection for you.

### **Demo 5 - Choosing the Right Technique**

**Note: We cover the OpenAI SDK properly in Class 13. The table below is a static walkthrough of the same three problems and the recommended technique for each.**

---

| Problem | Best technique | Key signal | Prompt structure |
|---|---|---|---|
| Translate 'good morning' into French. | Direct prompt | Single step, no intermediate reasoning needed - pure retrieval | `Translate '...' into French.` |
| A farmer has 17 sheep. All but 9 run away. How many are left? | Standard CoT | 'All but' requires careful parsing before arithmetic - a linguistic trap that causes direct answers to fail | `Solve step by step: ...` |
| Design a database schema for a hospital management system handling patients, doctors, appointments, and billing - including patients who see multiple doctors. | Reflection (or ToT) | Explicit list of requirements to satisfy; early decisions (e.g., many-to-many relationship) constrain later ones | Draft schema, then: `Check: does this satisfy all stated requirements? Revise if not.` |

---

The signals to look for:
- **Direct:** one lookup, no intermediate steps needed
- **CoT:** sequential steps where each depends on the previous
- **Self-consistency:** math or logic where wrong paths are plausible and you need confidence
- **Reflection:** explicit checklist of requirements you can verify
- **ToT:** early decisions constrain later options; backtracking has value


Look at which signals the model identifies. Translation = direct (one step, no reasoning needed). The sheep problem = CoT (the phrase "all but" requires careful parsing before arithmetic). The schema design = reflection or ToT (explicit requirements to satisfy, planning decisions that affect each other). This classification logic is what you should internalize - the problem structure tells you the technique.

## **PART 7 - All Four Techniques on the Same Problem**

**Note: We cover the OpenAI SDK properly in Class 13. The outputs below are static examples showing how each technique handles the same logic puzzle.**

---

**Puzzle:**

```
Four people - Arjun, Bhavna, Chetan, Divya - each prefer a different programming language:
Python, JavaScript, Java, or Go.

Clues:
1. Arjun does not use Python or JavaScript.
2. Bhavna uses a language that starts with 'J'.
3. Chetan and Divya do not use Go.
4. Bhavna does not use Java.
```

---

**Technique 1 - Direct answer:**

```
Arjun: Go, Bhavna: JavaScript, Chetan: Python, Divya: Java.
```
*(May be correct, but no work shown - no way to verify the reasoning or catch errors.)*

---

**Technique 2 - Standard CoT:**

```
Clue 2: Bhavna uses a language starting with 'J' - either JavaScript or Java.
Clue 4: Bhavna does not use Java. So Bhavna uses JavaScript.
Clue 1: Arjun does not use Python or JavaScript. Remaining for Arjun: Java or Go.
Clue 3: Chetan and Divya do not use Go.
  - If Arjun takes Java, then Go must go to Chetan or Divya - but clue 3 forbids that.
  - So Arjun takes Go.
Remaining: Python and Java for Chetan and Divya. No further constraints on these two.
Assign Chetan: Python, Divya: Java.

Answer: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java.
```

---

**Technique 3 - Self-consistency (3 paths):**

```
Path 1 final: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java
Path 2 final: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java
Path 3 final: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java

Majority vote: unanimous.
```
*(This puzzle is deterministic - all paths converge. Self-consistency adds most value on problems with numerical ambiguity where wrong paths produce plausible but different wrong answers.)*

---

**Technique 4 - Reflection:**

```
Draft: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java.

Critique - checking each clue:
1. Arjun does not use Python or JavaScript. Arjun=Go. Satisfied.
2. Bhavna uses a language starting with 'J'. Bhavna=JavaScript. Satisfied.
3. Chetan and Divya do not use Go. Chetan=Python, Divya=Java. Satisfied.
4. Bhavna does not use Java. Bhavna=JavaScript. Satisfied.
All clues satisfied. No revision needed.

Final: Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java.
```


This is the core comparison cell. For this puzzle, standard CoT and reflection should both reach the correct answer (Arjun=Go, Bhavna=JavaScript, Chetan=Python, Divya=Java - or similar valid assignment). The direct answer may be right or wrong depending on whether the model has seen similar puzzles. Self-consistency across 3 paths should converge. The useful observation here is not which one wins on this example - it is understanding why each technique is structured the way it is.

# **Key Takeaways**

1. Standard CoT works because intermediate reasoning tokens become part of the attention context, grounding each next step - not through any special reasoning mechanism, just through the normal transformer forward pass.

2. Self-consistency trades inference cost for reliability. Sample k paths at temperature > 0, take plurality answer. Works because correct reasoning paths converge, incorrect paths diverge.

3. Tree of Thought is for problems where early decisions constrain later options and backtracking has value. It runs multiple LLM calls per decision point to prune dead ends early.

4. Reflection exploits the model's asymmetry between generation and evaluation. A model that generates a buggy answer will often catch that bug when asked to review the same code.

5. The problem structure tells you the technique: sequential steps = CoT, accuracy-critical with budget = self-consistency, planning with branching = ToT, explicit checklist of requirements = reflection.

6. Reasoning models (o1/o3/Claude Extended Thinking) are CoT baked into training via RL. They automate technique selection for hard problems but cost more and run slower.

7. These techniques are not mutually exclusive. Self-consistency uses CoT internally. A full pipeline might use ToT for planning, CoT for each step, and reflection on the final output.

# **Quick Check**

1. A model gives a confident wrong answer to a word problem when prompted directly, but gets it right with CoT. Explain mechanically why the intermediate steps help - what changes in the transformer's computation?

2. You run self-consistency with k=5 on a math problem and get answers: [72, 75, 72, 72, 75]. What do you return? What if the split is [72, 75, 73, 71, 75]?

3. You are building a travel itinerary planner. A user gives 10 cities and 7 days with budget and preference constraints. Which technique fits best and why? What is the signal in this problem?

4. Why does reflection work better for code generation than for trivia questions? What property of code makes self-critique reliable?

5. What is the fundamental cost trade-off between standard CoT and self-consistency? Give a scenario where the extra cost of self-consistency is worth it and one where it is not.

6. You need a model to write a product requirements document that must include: problem statement, user stories, acceptance criteria, and out-of-scope items. Which technique - CoT or reflection - is better suited, and why?

7. Reasoning models like o1 outperform prompt-level CoT on hard math. What is the structural reason for this, given what you now know about how both work?

## **Quick Check - Answers**

---

*Attempt the questions above before reading these answers.*

---

**1.** CoT forces the model to generate intermediate reasoning steps as actual output tokens. Each step becomes part of the context for predicting the next token. The transformer attends to these intermediate computations when generating the final answer, effectively using the output stream as scratch space - impossible when the answer is forced in a single generation step.

**2.** First case: return 72 (appears 3 out of 5 - majority). Second case: return 75 (most frequent at 2 out of 5 with no majority). When there is no majority, use the most frequent; if still tied, the variance itself signals low model confidence on this input.

**3.** Reflection (or Plan-and-Execute) fits best. The signal is multiple interacting constraints that must all be satisfied simultaneously - dates, budget, preferences, and city order all interact. CoT generates one pass through the reasoning; reflection lets the model draft an itinerary, then explicitly verify each constraint and revise. Pure CoT cannot catch its own constraint violations.

**4.** Code has a formal correctness criterion - you can execute it and verify output. The model can reason about whether the code does what it claims. Trivia questions have no self-check: if the model generated a wrong fact, it has no mechanism to detect the error because the "answer" cannot be verified from first principles within the model's reasoning.

**5.** Self-consistency runs the same prompt k times and takes a majority vote, costing k API calls. Worth it: high-stakes decisions where answer variance matters (medical, financial, legal). Not worth it: latency-sensitive applications (real-time chat) or tasks where single-pass quality is already sufficient.

**6.** Reflection is better suited. The model can draft the PRD, then explicitly check whether each required section is present and complete - a clear, verifiable self-check criterion. CoT would write it in one pass but cannot catch missing sections. The review loop maps directly onto document completeness requirements.

**7.** Reasoning models (o1, o3) perform chain-of-thought internally as latent tokens that influence output but are not shown. They have far greater reasoning budgets and are trained specifically on reasoning tasks. Prompt-level CoT is limited by context window and the model's surface-level reasoning quality - it cannot match a model trained to reason deeply at the pre-training and fine-tuning level.

## Extension: Tree of Thought (Optional)

The demos below show ToT in action for a planning problem. Run these after completing the Quick Check. This pattern illustrates how multi-step branching and evaluation work - you will build on multi-step reasoning patterns throughout the course.


**Note: We cover the OpenAI SDK properly in Class 13. The three steps below are static examples showing the generate-evaluate-expand loop of Tree of Thought.**

---

**Problem:**

```
Plan a 3-day tech conference for 200 attendees.
Budget: 5 lakh rupees total.
Must include: keynotes, workshops, networking sessions.
Constraint: day 3 must end by 4 PM for travel.
```

---

**Step 1 - Generate candidate approaches (breadth):**

```
Prompt: Generate 3 different high-level planning approaches. Number them 1, 2, 3.

Approach 1 - Theme-first:
Organise each day around a single theme (Day 1: Fundamentals, Day 2: Applied AI,
Day 3: Future and Industry). Keynotes, workshops, and networking all align with that theme.
Single track, cost-efficient.

Approach 2 - Audience-split:
Morning sessions for beginners, afternoon for advanced practitioners.
Two parallel tracks require more venue space and AV budget.
Day 3 runs only until early afternoon to respect the 4 PM constraint.

Approach 3 - Keynote-heavy:
Two flagship keynotes per day bookend the schedule.
Workshops are 90-minute deep dives in between.
Networking during extended lunch breaks. Risk of attendee fatigue.
```

---

**Step 2 - Evaluate each candidate:**

```
Prompt: Evaluate each approach against budget feasibility, all requirements,
and the day 3 constraint. Score 1-10 and recommend the best.

Approach 1 (Theme-first): 8/10
  - Budget: single-track is most cost-efficient. Feasible.
  - Requirements: all three component types fit naturally.
  - Day 3: easy to schedule last session by 3:30 PM.

Approach 2 (Audience-split): 5/10
  - Budget: two parallel tracks doubles AV and facilitator costs. Tight.
  - Requirements: satisfied but complex to coordinate.
  - Day 3: manageable.

Approach 3 (Keynote-heavy): 6/10
  - Budget: high-profile speakers raise costs. Risky.
  - Requirements: networking is informal only - weaker on that dimension.
  - Day 3: satisfied.

Recommendation: Approach 1 (Theme-first).
```

---

**Step 3 - Expand the best branch:**

```
Prompt: Take Approach 1. Create a detailed day-by-day schedule with time slots,
estimated cost per component, and running total vs budget.

DAY 1 - Fundamentals
  09:00-09:30  Registration and welcome kit         15,000
  09:30-10:30  Keynote: Foundations of Modern AI    40,000
  10:30-12:30  Workshop: Prompt Engineering 101     20,000
  12:30-13:30  Networking lunch                     30,000
  13:30-15:30  Workshop: RAG Fundamentals           20,000
  15:30-17:00  Panel discussion and Q&A             15,000
  Day 1 subtotal: 1,40,000  |  Running total: 1,40,000 / 5,00,000

DAY 2 - Applied AI (similar structure)             ~1,50,000
  Running total: 2,90,000 / 5,00,000

DAY 3 - Future and Industry (ends by 4 PM)
  09:00-11:00  Morning workshop                     20,000
  11:00-12:30  Keynote: AI in Industry              40,000
  12:30-13:30  Networking lunch                     30,000
  13:30-15:30  Closing panel and awards             15,000
  15:30        Event ends (within 4 PM constraint)
  Day 3 subtotal: 1,05,000  |  Running total: 3,95,000 / 5,00,000

Venue, AV, contingency: ~80,000
Grand total: ~4,75,000 - within the 5 lakh budget.
```


Notice the structure: generate multiple candidates first, evaluate without committing, then expand only the winner. This is the core ToT loop in three LLM calls. In a full implementation you would iterate - expand the winner, generate sub-candidates for the next decision point, evaluate again. The trade-off vs standard CoT is clear: 3x the API calls, but the final plan is constrained by explicit evaluation at each step rather than committing to the first direction that sounded reasonable.